In [14]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name())

cuda NVIDIA GeForce MX350


In [15]:
from torch import nn
from torch.nn import Module

class Model(Module):
    def __init__(self, num_features):
        super().__init__() # calling base class constructor is maindatory, everything happening under the hood gets initialized and managed by base class (parameter initialization and its state, BatchNorm state, cpu → gpu transfe, etc)

        self.network = nn.Sequential(
            # two types of models → sequential, functional
            nn.Linear(in_features = num_features, out_features = 3),
            nn.ReLU(),
            nn.Linear(in_features = 3, out_features = 1), 
            # nn.Sigmoid() → no need to use any activation function on output layer → loss function automatically applies it before actually calculating the loss
        )
        # the model parameters are randomly initialized in float32 datatype → your input features must be in that same datatype, not even float64 or float16 allowed (implicite conversion does not happen)

    def forward(self, features):
        out = self.network(features) # magic method __call__ → just pass the input features to object `obj(input)`
        return out

If you want to train your model on GPU, your model and data must be shifted on GPU memory.

In [16]:
features = torch.rand(10, 5, requires_grad=False).to(device=device)
model = Model(features.shape[1]).to(device=device)

model(features) # forward method through `__call__` magic method

tensor([[-0.3185],
        [-0.3321],
        [-0.5607],
        [-0.3858],
        [-0.3909],
        [-0.4055],
        [-0.4490],
        [-0.4356],
        [-0.3188],
        [-0.3030]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [17]:
# model architecture
from torchinfo import summary
summary(model, input_size=(10, 5), device=device)

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00